# AICUP 2025 春季賽 — 桌球智慧球拍資料精準分析

本 notebook 從六軸 IMU 揮拍訊號（每秒 85 筆）預測選手四項屬性：`gender`、`hold racket handed`、`play years`、`level`。
每個 cell 都對應 `README.md` 的章節，方便對照閱讀。

| Notebook 區塊 | 對應 README 章節 | 重點 |
| --- | --- | --- |
| 1. Setup & Global Config | Setup / Running | 套件、全域常數、TSFEL 設定 |
| 2. Feature Extraction | Model overview (Features) | TSFEL per-segment 特徵 |
| 3. Aggregation & Selection | Model overview (Selection) | 7 統計量聚合、VarianceThreshold |
| 4. Per-fold Train & Evaluate | Evaluation protocol | 單折 CatBoost 訓練與評分 |
| 5. Main Pipeline | How it works | Optuna 調參、模型儲存、測試集預測 |
| 6. Run | Running | 一鍵執行整條 pipeline |

**整體 Pipeline**：`Raw .txt` → `TSFEL (per-segment 992 dims)` → `Aggregate (7 statistics, 6,944 dims)` → `VarianceThreshold` → `SelectKBest (每個 fold 內擬合, k tuned by Optuna)` → `CatBoost GPU (per target × 4)` → `Submission probabilities`。

> 註：`SelectKBest` 僅套用於 `gender` / `play years` / `level`；`hold racket handed` 使用 VarianceThreshold 後的全部特徵。完整流程圖見 `README.md` 的 How it works 章節。


## 1. Setup & Global Config

對應 README 的 **Setup** 與 **Running**。此區塊完成三件事：

- **套件導入**：`numpy` / `pandas`、`sklearn`（`GroupKFold`、`KNNImputer`、`MinMaxScaler`、`VarianceThreshold`、`SelectKBest`）、`catboost`、`optuna`、`tsfel`、`joblib`。
- **logging 與 warning 設定**：以 `logging` 輸出各步驟進度，並抑制 TSFEL / CatBoost / pandas 等套件的雜訊 warning。
- **全域常數**：
  - `NUM_TOTAL_SEGMENTS_PER_FILE = 27`：每筆資料等分成 27 個 swing segment。
  - `TARGET_COLS`（四項 target）、`BINARY_TARGETS`（`gender`、`hold racket handed`）。
  - `N_SPLITS_GROUPKFOLD = 5`、`OPTUNA_N_TRIALS = 75`、`TSFEL_SAMPLING_FREQ = 85`（官方說明感測器每秒 85 筆）。
  - `USE_SAVED_MODELS = True`：若已存在訓練好的 artifact 則直接載入、跳過訓練。
  - `RANDOM_SEED` / `RNG`：input noise 使用固定 seed 的亂數；`N_JOBS_TSFEL`：TSFEL 抽取的平行 process 數。
  - `tsfel_cfg`：取得 TSFEL 全 domain（temporal + spectral + statistical）特徵設定，並停用欄位名稱會隨段長改變的 `Spectrogram mean coefficient`；失敗時退回 `TSFEL_MINIMAL_CFG`。
  - `AGGREGATION_METHODS_MAP`：將 27 個 segment 聚合成單筆資料特徵時使用的 **7 種統計量**（mean / std / median / min / max / skew / kurtosis）。


In [ ]:
# 導入必要的套件
from pathlib import Path
import numpy as np
import pandas as pd
import csv
from sklearn.model_selection import GroupKFold
from catboost import CatBoostClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    balanced_accuracy_score,
)
from sklearn.impute import KNNImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight  # 用於計算類別權重
import warnings
import logging
from typing import List, Dict, Tuple, Any, Optional, Union
import numpy.typing as npt
import optuna
import tsfel
import joblib
from joblib import Parallel, delayed
import json

# --- 設定日誌記錄 ---
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(filename)s:%(lineno)d] - %(message)s",
)
logger = logging.getLogger(__name__)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- 抑制警告 ---
warnings.filterwarnings("ignore", category=FutureWarning, module="pandas.core.frame")
warnings.filterwarnings("ignore", category=UserWarning, module="catboost")
warnings.filterwarnings(
    "ignore", category=RuntimeWarning, message="invalid value encountered in divide"
)
warnings.filterwarnings("ignore", category=RuntimeWarning, message="Mean of empty slice")
warnings.filterwarnings("ignore", message="", category=UserWarning, module="tsfel")
warnings.filterwarnings("ignore", message="", category=RuntimeWarning, module="tsfel")
warnings.filterwarnings(
    "ignore", category=RuntimeWarning, message="invalid value encountered in cast"
)


# --- 全域常數 ---
NUM_TOTAL_SEGMENTS_PER_FILE = 27
TARGET_COLS = ["gender", "hold racket handed", "play years", "level"]
BINARY_TARGETS = ["gender", "hold racket handed"]
N_SPLITS_GROUPKFOLD = 5
OPTUNA_N_TRIALS = 75
TSFEL_SAMPLING_FREQ = 85  # 官方說明：感測器每秒 85 筆

MODELS_DIR = Path(".") / "trained_models_catboost_v7"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
USE_SAVED_MODELS = True
RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)  # 固定 input noise 的亂數，結果可重現
N_JOBS_TSFEL = -1  # TSFEL 特徵抽取的平行 process 數 (-1 = 全部 CPU 核心)

TSFEL_MINIMAL_CFG = {
    "statistical": {
        "mean": None,
        "median": None,
        "std": None,
        "var": None,
        "skew": None,
        "kurtosis": None,
        "max": None,
        "min": None,
        "rms": None,
        "abs_energy": None,
        "mean_abs_deviation": None,
        "median_abs_deviation": None,
    },
    "temporal": {
        "mean_abs_diff": None,
        "mean_diff": None,
        "distance": None,
        "slope": None,
        "zero_crossings": None,
        "positive_turnings": None,
        "negative_turnings": None,
        "peak_to_peak_distance": None,
    },
}
try:
    tsfel_cfg = tsfel.get_features_by_domain()
    if not tsfel_cfg:
        logger.warning("TSFEL get_features_by_domain() 返回空配置，使用預定義的最小配置。")
        tsfel_cfg = TSFEL_MINIMAL_CFG
except Exception as e_tsfel_cfg:
    logger.error(f"無法獲取 TSFEL 預設配置: {e_tsfel_cfg}. 使用預定義的最小配置。")
    tsfel_cfg = TSFEL_MINIMAL_CFG

# 'Spectrogram mean coefficient' 的欄位以頻率命名 (例如 _0.83Hz)，頻率點取決於每段揮拍的長度。
# 各檔案段長不同 (訓練集有 119 種)，對齊到第一個檔案的欄位後，九成以上的值會是 NaN，因此停用此特徵。
if "spectral" in tsfel_cfg and "Spectrogram mean coefficient" in tsfel_cfg["spectral"]:
    tsfel_cfg["spectral"]["Spectrogram mean coefficient"]["use"] = "no"

AGGREGATION_METHODS_MAP: Dict[str, str] = {
    "_mean": "mean",
    "_std": "std",
    "_median": "median",
    "_min": "min",
    "_max": "max",
    "_skew": "skew",
    "_kurt": "kurtosis",
}

## 2. Feature Extraction with TSFEL

對應 README 的 **Model overview**（Features）。每筆 `.txt` 平均切成 27 個 swing segment（未使用官方的 `cut_point`），每個 segment 對 8 條訊號抽取 TSFEL 時序特徵：

- `calculate_euclidean_magnitude`：由三軸合成 L2 magnitude（加速度合成 `acc_mag`、角速度合成 `gyro_mag`），補足 6 軸原始訊號之外的 2 條訊號。
- `extract_features_for_one_signal_with_tsfel`：對單一訊號跑 `tsfel.time_series_features_extractor`；整批抽取失敗時（例如很短的資料）改為逐一特徵計算，只略過出錯的特徵。
- `extract_tsfel_features_for_segment`：對 1 個 segment 的 6 軸 + 2 magnitude 共 8 條訊號各自抽取 TSFEL 特徵並橫向拼接，**per-segment 992 維**（每條訊號 124 維）。
- `data_generate_tsfel_segment_features_from_raw`：讀取原始 `.txt`、用 `np.linspace` 等分成 27 段、逐段抽特徵，多個 process 平行處理，並把每筆資料的結果 **cache 成一個 CSV**（已存在則略過，支援快速重跑）。


In [ ]:
# --- 特徵提取、聚合、選擇函數 (與之前版本相同) ---
def calculate_euclidean_magnitude(
    data_array_n_axes: npt.NDArray[np.float64],
) -> npt.NDArray[np.float64]:
    if (
        not isinstance(data_array_n_axes, np.ndarray)
        or data_array_n_axes.ndim != 2
        or data_array_n_axes.shape[0] == 0
    ):
        return np.array([], dtype=np.float64)
    return np.sqrt(np.sum(data_array_n_axes**2, axis=1)).astype(np.float64)


def extract_features_for_one_signal_with_tsfel(
    signal_series: pd.Series, sampling_freq: int, feature_config: Dict
) -> Optional[pd.DataFrame]:
    min_len_tsfel = max(5, sampling_freq // 10 if sampling_freq > 0 else 5)
    if (
        signal_series.empty
        or not isinstance(signal_series, pd.Series)
        or signal_series.shape[0] < min_len_tsfel
    ):
        return None
    try:
        features_df = tsfel.time_series_features_extractor(
            feature_config, signal_series.astype(float), fs=sampling_freq, verbose=0
        )
        return features_df.reset_index(drop=True) if not features_df.empty else None
    except Exception:
        pass
    # 很短的資料 (例如提早中止的有氧測試) 每段只有約 10 個點，LPCC 等少數特徵會報錯；
    # 此時改為逐一特徵計算，只略過出錯的特徵；欄位名稱與正常長度的段相同，缺少的欄位之後會被 reindex 成 NaN。
    parts = []
    for domain, domain_features in feature_config.items():
        for feature_name, feature_spec in domain_features.items():
            if feature_spec.get("use", "yes") != "yes":
                continue
            try:
                parts.append(
                    tsfel.time_series_features_extractor(
                        {domain: {feature_name: feature_spec}},
                        signal_series.astype(float),
                        fs=sampling_freq,
                        verbose=0,
                    )
                )
            except Exception:
                continue
    return pd.concat(parts, axis=1).reset_index(drop=True) if parts else None


def extract_tsfel_features_for_segment(
    segment_np: npt.NDArray[np.float64],
    sampling_freq: int = TSFEL_SAMPLING_FREQ,
    feature_config: Dict = tsfel_cfg,
) -> Optional[pd.DataFrame]:
    min_len_segment = 4
    if segment_np.shape[0] < min_len_segment or segment_np.shape[1] < 6:
        return None
    all_extracted_features_list = []
    axis_names_base = ["ax", "ay", "az", "gx", "gy", "gz"]
    for i in range(6):
        axis_signal = pd.Series(segment_np[:, i].astype(np.float64))
        axis_features = extract_features_for_one_signal_with_tsfel(
            axis_signal, sampling_freq, feature_config
        )
        if axis_features is not None and not axis_features.empty:
            all_extracted_features_list.append(
                axis_features.add_prefix(f"{axis_names_base[i]}_tsfel_")
            )
    acc_mag_arr = calculate_euclidean_magnitude(segment_np[:, :3])
    gyro_mag_arr = calculate_euclidean_magnitude(segment_np[:, 3:6])
    for mag_name, mag_data_arr in {"acc_mag": acc_mag_arr, "gyro_mag": gyro_mag_arr}.items():
        if mag_data_arr.ndim == 1 and mag_data_arr.shape[0] >= min_len_segment:
            mag_signal = pd.Series(mag_data_arr)
            mag_features = extract_features_for_one_signal_with_tsfel(
                mag_signal, sampling_freq, feature_config
            )
            if mag_features is not None and not mag_features.empty:
                all_extracted_features_list.append(mag_features.add_prefix(f"{mag_name}_tsfel_"))
    if not all_extracted_features_list:
        return None
    try:
        return pd.concat(all_extracted_features_list, axis=1).astype(np.float32)
    except Exception as e_concat:
        logger.error(f"合併 TSFEL 特徵時出錯: {e_concat}")
        return None


def _read_raw_imu_file(file_path: Path) -> List[List[float]]:
    rows: List[List[float]] = []
    with open(file_path, "r") as f:
        for line in f:
            parts = [p for p in line.strip().split(" ") if p]
            if len(parts) >= 6:
                try:
                    rows.append([float(n) for n in parts[:6]])
                except ValueError:
                    continue
    return rows


def _process_one_raw_file(
    file_path: Path, tar_dir: Path, known_feature_names: Optional[List[str]]
) -> Tuple[bool, Optional[List[str]]]:
    """處理單一 .txt：切成 27 段、逐段抽 TSFEL 特徵、存成一個 CSV。回傳 (是否成功, 特徵欄位名)。"""
    warnings.filterwarnings("ignore")
    output_csv_path = tar_dir / f"{file_path.stem}.csv"
    try:
        all_data_raw = _read_raw_imu_file(file_path)
    except Exception as e:
        logger.error(f"讀取原始數據檔案 {file_path.name} 失敗: {e}")
        return False, known_feature_names
    if not all_data_raw:
        return False, known_feature_names
    feature_names = known_feature_names
    swing_indices = np.linspace(0, len(all_data_raw), NUM_TOTAL_SEGMENTS_PER_FILE + 1, dtype=int)
    file_segments_list = []
    for i in range(NUM_TOTAL_SEGMENTS_PER_FILE):
        segment_raw_data = all_data_raw[swing_indices[i] : swing_indices[i + 1]]
        seg_df = None
        if len(segment_raw_data) >= 4:
            seg_df = extract_tsfel_features_for_segment(
                np.array(segment_raw_data, dtype=np.float64)
            )
        if seg_df is not None and not seg_df.empty:
            if feature_names is None:
                feature_names = list(seg_df.columns)
            file_segments_list.append(
                seg_df.reindex(columns=feature_names, fill_value=np.nan).astype(np.float32)
            )
        elif feature_names:
            file_segments_list.append(
                pd.DataFrame(np.nan, index=[0], columns=feature_names).astype(np.float32)
            )
    if len(file_segments_list) == NUM_TOTAL_SEGMENTS_PER_FILE:
        try:
            pd.concat(file_segments_list, ignore_index=True).to_csv(output_csv_path, index=False)
            return True, feature_names
        except Exception as e:
            logger.error(f"保存 TSFEL 特徵檔案 {output_csv_path.name} 失敗: {e}")
    elif file_segments_list:
        logger.warning(
            f"檔案 {file_path.name} 僅處理了 {len(file_segments_list)}/{NUM_TOTAL_SEGMENTS_PER_FILE} 個片段。"
        )
    return False, feature_names


def data_generate_tsfel_segment_features_from_raw(
    datapath_str: str, tar_dir_str: str, known_feature_names: Optional[List[str]] = None
) -> Optional[List[str]]:
    datapath = Path(datapath_str)
    tar_dir = Path(tar_dir_str)
    tar_dir.mkdir(parents=True, exist_ok=True)
    pathlist_txt = sorted(datapath.glob("*.txt"))
    if not pathlist_txt:
        logger.warning(f"在 {datapath} 中找不到 .txt 檔案。")
        return known_feature_names
    current_run_feature_names: Optional[List[str]] = known_feature_names
    todo: List[Path] = []
    for file_path in pathlist_txt:
        output_csv_path = tar_dir / f"{file_path.stem}.csv"
        if output_csv_path.exists():
            if current_run_feature_names is None:
                try:
                    current_run_feature_names = list(pd.read_csv(output_csv_path, nrows=0).columns)
                except Exception as e:
                    logger.warning(f"無法從現有檔案 {output_csv_path.name} 讀取特徵名: {e}")
        else:
            todo.append(file_path)
    logger.info(
        f"正在為 {datapath_str} 生成 TSFEL 片段特徵至 {tar_dir_str}：{len(todo)}/{len(pathlist_txt)} 個檔案需要抽取 (n_jobs={N_JOBS_TSFEL})..."
    )
    # 先序列處理檔案直到確定特徵欄位順序，其餘檔案平行處理並對齊這組欄位
    while todo and current_run_feature_names is None:
        _, current_run_feature_names = _process_one_raw_file(todo.pop(0), tar_dir, None)
    if todo:
        Parallel(n_jobs=N_JOBS_TSFEL, verbose=5)(
            delayed(_process_one_raw_file)(fp, tar_dir, current_run_feature_names) for fp in todo
        )
    processed_files_count = sum(1 for fp in pathlist_txt if (tar_dir / f"{fp.stem}.csv").exists())
    logger.info(
        f"TSFEL 片段特徵生成完成。共處理/檢查 {processed_files_count}/{len(pathlist_txt)} 個檔案。"
    )
    return current_run_feature_names

## 3. Aggregation & Selection

對應 README 的 **Model overview**（Selection）。

- `aggregate_segment_features`：把一筆資料的 27 個 segment（per-segment 992 維）以 **7 種統計量**聚合成單一特徵向量：
  `6,944 維 = 992 × 7 (mean / std / median / min / max / skew / kurtosis)`。
- `select_features_basic`：以 `VarianceThreshold(0.01)` 全域移除近常數欄位，同時處理 inf / 全 NaN 欄位。此步驟不使用標籤。

> 注意：`SelectKBest(f_classif)` 的 per-target 篩選並非在此處執行，而是在第 5 區塊的 Optuna 流程中，**於每個 fold 內只用訓練資料擬合**，套用於 `gender` / `play years` / `level`（`k` 由 Optuna 搜尋）。


In [ ]:
def select_features_basic(
    X_df: pd.DataFrame, variance_threshold_val: float = 0.01
) -> Tuple[pd.DataFrame, List[str]]:
    logger.info(f"全局基礎特徵選擇前 - 形狀: {X_df.shape}")
    if X_df.empty:
        logger.warning("輸入 DataFrame 為空，跳過全局基礎特徵選擇。")
        return X_df, []
    X_df_c = X_df.copy()
    X_df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
    nan_cols_all_nan = X_df_c.columns[X_df_c.isnull().all()].tolist()
    if nan_cols_all_nan:
        X_df_c = X_df_c.drop(columns=nan_cols_all_nan)
        logger.info(f"移除了 {len(nan_cols_all_nan)} 個所有值均為 NaN 的列。")
    if X_df_c.empty:
        logger.warning("移除全 NaN 列後 DataFrame 為空。")
        return X_df_c, []
    num_cols = X_df_c.select_dtypes(include=np.number).columns
    if not num_cols.empty:
        X_df_c[num_cols] = X_df_c[num_cols].fillna(X_df_c[num_cols].median())
        X_df_c[num_cols] = X_df_c[num_cols].fillna(0)
    if X_df_c.empty:
        return X_df_c, list(X_df_c.columns)
    num_df_for_variance = X_df_c.select_dtypes(include=np.number)
    if num_df_for_variance.empty:
        logger.warning("沒有數值型特徵可供 VarianceThreshold 處理。")
        return X_df_c, list(X_df_c.columns)
    selector = VarianceThreshold(threshold=variance_threshold_val)
    try:
        selector.fit(num_df_for_variance)
        sel_mask = selector.get_support()
        if not np.any(sel_mask):
            logger.warning("VarianceThreshold 未選中任何特徵。")
            return X_df, list(X_df.columns)
        selected_numerical_features = num_df_for_variance.columns[sel_mask].tolist()
        non_numerical_features = X_df_c.select_dtypes(exclude=np.number).columns.tolist()
        final_selected_features = selected_numerical_features + non_numerical_features
        result_df = X_df[final_selected_features]
        logger.info(f"全局基礎特徵選擇後 - 形狀: {result_df.shape}")
        return result_df, final_selected_features
    except Exception as e:
        logger.error(f"VarianceThreshold 執行失敗: {e}.")
        return X_df_c, list(X_df_c.columns)


def aggregate_segment_features(
    segment_features_df: pd.DataFrame, aggregation_methods_map: Dict[str, str]
) -> pd.Series:
    if segment_features_df.empty:
        return pd.Series(dtype="float64")
    num_df = segment_features_df.select_dtypes(include=np.number)
    if num_df.empty:
        return pd.Series(dtype="float64")
    aggregated_parts = []
    original_col_names = list(num_df.columns)
    for suffix, func_str in aggregation_methods_map.items():
        try:
            agg_values = num_df.agg(func_str, axis=0)
            if not isinstance(agg_values, pd.Series):
                agg_values = pd.Series(agg_values, index=original_col_names)
            agg_values.index = [f"{name}{suffix}" for name in original_col_names]
            aggregated_parts.append(agg_values)
        except Exception:
            aggregated_parts.append(
                pd.Series(
                    [np.nan] * len(original_col_names),
                    index=[f"{name}{suffix}" for name in original_col_names],
                )
            )
    if not aggregated_parts:
        return pd.Series(dtype="float64")
    final_aggregated_series = pd.concat(aggregated_parts)
    final_aggregated_series.replace([np.inf, -np.inf], np.nan, inplace=True)
    return final_aggregated_series.astype(np.float32)

## 4. Per-fold Train & Evaluate

對應 README 的 **Evaluation protocol**。`train_evaluate_model_cv_fold` 是 Optuna 在每個 `GroupKFold` fold 內呼叫的核心函式：

- **前處理**：`KNNImputer(n_neighbors=5)` 補值 → `MinMaxScaler` 縮放（皆只 `fit` 在 train fold，再 `transform` validation fold，避免 leakage）。
- **input noise**：`add_input_noise` 為真時對縮放後的訓練資料加 Gaussian noise（固定 seed），**噪聲只施加於訓練資料**。
- **模型**：`CatBoostClassifier`（`task_type='GPU'`），以 validation fold 作 `eval_set` 並啟用 early stopping，記錄最佳迭代數供最終模型使用。
- **評估指標**：binary target 用 `roc_auc`；multi-class target 用 micro / macro OvR ROC AUC；另外記錄 `f1`、`mcc`、`balanced_accuracy` 作參考。


In [ ]:
# --- CatBoost 模型訓練與評估函式 ---
def train_evaluate_model_cv_fold(
    trial: Optional[optuna.Trial],
    X_train_fold: pd.DataFrame,
    y_train_fold: npt.NDArray[np.int_],
    X_val_fold: pd.DataFrame,
    y_val_fold: npt.NDArray[np.int_],
    target_name: str,
    is_binary: bool,
    label_encoder_target: LabelEncoder,
    fold_num: int,
    catboost_trial_params: Dict[str, Any],
) -> Dict[str, float]:
    n_neighbors_knn = min(5, X_train_fold.shape[0] - 1 if X_train_fold.shape[0] > 1 else 1)
    if n_neighbors_knn < 1:
        n_neighbors_knn = 1
    imputer = KNNImputer(n_neighbors=n_neighbors_knn)
    X_train_fold_cols = list(X_train_fold.columns)
    X_val_fold_cols = list(X_val_fold.columns)
    try:
        X_train_fold_imputed_arr = imputer.fit_transform(X_train_fold)
        X_train_fold_imputed = pd.DataFrame(
            X_train_fold_imputed_arr, columns=X_train_fold_cols
        ).astype(np.float32)
        if not X_val_fold.empty:
            X_val_fold_imputed_arr = imputer.transform(X_val_fold)
            X_val_fold_imputed = pd.DataFrame(
                X_val_fold_imputed_arr, columns=X_val_fold_cols
            ).astype(np.float32)
        else:
            X_val_fold_imputed = pd.DataFrame(columns=X_val_fold_cols, dtype=np.float32)
    except ValueError as e:
        logger.error(f"Fold {fold_num+1} Target {target_name}: KNN 插補失敗 - {e}.")
        return {
            k: np.nan
            for k in [
                "roc_auc",
                "roc_auc_micro_ovr",
                "roc_auc_macro_ovr",
                "f1_macro",
                "f1_weighted",
                "mcc",
                "balanced_accuracy",
            ]
        }
    scaler = MinMaxScaler()
    X_train_scaled_no_noise = scaler.fit_transform(X_train_fold_imputed).astype(np.float32)
    if not X_val_fold_imputed.empty:
        X_val_scaled = scaler.transform(X_val_fold_imputed).astype(np.float32)
    else:
        X_val_scaled = np.array([]).reshape(0, X_train_scaled_no_noise.shape[1]).astype(np.float32)
    X_fit_data = X_train_scaled_no_noise.copy()
    if (
        catboost_trial_params.get("add_input_noise", False)
        and catboost_trial_params.get("noise_level", 0.0) > 0
    ):
        noise = RNG.normal(0, catboost_trial_params["noise_level"], X_train_scaled_no_noise.shape)
        X_fit_data = (X_train_scaled_no_noise + noise).astype(np.float32)

    model_specific_params = {
        k: v
        for k, v in catboost_trial_params.items()
        if k
        not in [
            "add_input_noise",
            "noise_level",
            f"k_selectkbest_{target_name}",
            f"use_class_weights_{target_name}",
        ]
    }

    clf = CatBoostClassifier(**model_specific_params)
    eval_set_for_fit = None
    if X_val_scaled.shape[0] > 0 and y_val_fold.shape[0] > 0:
        eval_set_for_fit = (X_val_scaled, y_val_fold)
    else:
        if "early_stopping_rounds" in model_specific_params:
            del model_specific_params["early_stopping_rounds"]
    try:
        clf.fit(X_fit_data, y_train_fold, eval_set=eval_set_for_fit, verbose=0)
    except optuna.exceptions.TrialPruned:
        raise
    except Exception as e:
        logger.error(f"Fold {fold_num+1} Target {target_name}: CatBoost 模型訓練失敗 - {e}")
        return {
            k: np.nan
            for k in [
                "roc_auc",
                "roc_auc_micro_ovr",
                "roc_auc_macro_ovr",
                "f1_macro",
                "f1_weighted",
                "mcc",
                "balanced_accuracy",
            ]
        }
    scores = {
        "best_iteration": clf.get_best_iteration()
    }  # 記錄 early stopping 的最佳迭代數，給最終模型使用
    if eval_set_for_fit is None or X_val_scaled.shape[0] == 0:
        return {
            k: np.nan
            for k in [
                "roc_auc",
                "roc_auc_micro_ovr",
                "roc_auc_macro_ovr",
                "f1_macro",
                "f1_weighted",
                "mcc",
                "balanced_accuracy",
            ]
        }
    try:
        predicted_proba_val = clf.predict_proba(X_val_scaled)
        predicted_labels_val = clf.predict(X_val_scaled).flatten()
        y_val_fold_flat = y_val_fold.flatten()
        all_possible_labels_encoded = np.arange(len(label_encoder_target.classes_))
        y_val_unique_counts = np.unique(y_val_fold_flat, return_counts=True)
        if is_binary:
            if len(y_val_unique_counts[0]) < 2:
                scores["roc_auc"] = np.nan
            else:
                scores["roc_auc"] = roc_auc_score(y_val_fold_flat, predicted_proba_val[:, 1])
        else:
            if predicted_proba_val.shape[1] != len(all_possible_labels_encoded):
                logger.warning(f"Fold {fold_num+1} Target {target_name}: 預測概率列數不匹配。")
            try:
                scores["roc_auc_micro_ovr"] = roc_auc_score(
                    y_val_fold_flat,
                    predicted_proba_val,
                    average="micro",
                    multi_class="ovr",
                    labels=all_possible_labels_encoded,
                )
            except ValueError:
                scores["roc_auc_micro_ovr"] = np.nan
            try:
                scores["roc_auc_macro_ovr"] = roc_auc_score(
                    y_val_fold_flat,
                    predicted_proba_val,
                    average="macro",
                    multi_class="ovr",
                    labels=all_possible_labels_encoded,
                )
            except ValueError:
                scores["roc_auc_macro_ovr"] = np.nan
        scores["f1_macro"] = f1_score(
            y_val_fold_flat,
            predicted_labels_val,
            average="macro",
            labels=all_possible_labels_encoded,
            zero_division=0,
        )
        scores["f1_weighted"] = f1_score(
            y_val_fold_flat,
            predicted_labels_val,
            average="weighted",
            labels=all_possible_labels_encoded,
            zero_division=0,
        )
        scores["mcc"] = matthews_corrcoef(y_val_fold_flat, predicted_labels_val)
        scores["balanced_accuracy"] = balanced_accuracy_score(y_val_fold_flat, predicted_labels_val)
    except Exception as e:
        logger.error(f"Fold {fold_num+1} Target {target_name}: 模型評估失敗 - {e}")
        [
            scores.update({k_eval: np.nan})
            for k_eval in [
                "roc_auc",
                "roc_auc_micro_ovr",
                "roc_auc_macro_ovr",
                "f1_macro",
                "f1_weighted",
                "mcc",
                "balanced_accuracy",
            ]
        ]
    return scores

## 5. Main Pipeline

`main()` 串起整條 pipeline，對應 README 的 **How it works**。內部步驟：

- **步驟 1**：生成或載入 TSFEL segment 特徵（train / test，已 cache 則略過）。
- **步驟 2**：讀取 `train_info.csv`，取得 `player_id` 與四項 target 標籤。
- **步驟 3**：載入並以 7 統計量聚合 segment 特徵 → 特徵矩陣 `X=(1955, 6944)`。
- **步驟 3.5**：`VarianceThreshold` 全域移除近常數欄位。
- **步驟 4 & 6（每個 target 各一次）**：
  - 若 `USE_SAVED_MODELS=True` 且 artifact 齊全 → 直接載入並跳過訓練。
  - 否則用 **Optuna TPE** 跑 75 trials（搜尋 CatBoost 超參、`SelectKBest` 的 `k`、input noise、不平衡處理）；以 **`player_id` 分組的 5-fold `GroupKFold`** 評估，確保同選手不跨 train / validation。`SelectKBest` 在每個 fold 內只用訓練資料擬合。
  - 不平衡處理：`gender` 搜尋 `scale_pos_weight`；`play years` / `level` 可選用 fold-local balanced class weights。
  - 成本控制：每個 fold 結束後回報目前平均 AUC，`MedianPruner` 提前終止劣質 trial；每個 study 上限 2.5 小時 wall-clock。
  - **最終模型**：用全部訓練資料訓練，樹的數量取 CV 各 fold early stopping 最佳迭代數的平均。
  - **Persistence**：每個 target 存四項 artifact（`.cbm` 模型、`imputer`、`scaler`、`best_params_meta.json`）。
- **步驟 7**：對測試集逐筆聚合特徵、套用各 target 的模型輸出機率（不加雜訊），機率取到小數點後四位寫出 submission。
  - **Fallback**：測試資料的特徵檔缺少、為空或無法讀取時，採預設機率（binary → `0.5`，multi-class → `1/n_classes`）。

> Optuna Best CV 分數是在挑選參數的同一組 fold 上算出來的，仍然偏樂觀；要得到無偏的估計，需要把選出的參數放到另一組新的 fold 切分上重新評估。


In [ ]:
# --- 主函數 (main) ---
def main() -> None:
    base_data_dir = Path(".")
    tabular_train_segments_dir = base_data_dir / "tabular_data_train_tsfel_catboost_gpu_v7"
    tabular_test_segments_dir = base_data_dir / "tabular_data_test_tsfel_catboost_gpu_v7"
    train_data_raw_path = base_data_dir / "39_Training_Dataset" / "train_data"
    train_info_path = base_data_dir / "39_Training_Dataset" / "train_info.csv"
    test_data_raw_path = base_data_dir / "39_Test_Dataset" / "test_data"
    test_info_path = base_data_dir / "39_Test_Dataset" / "test_info.csv"
    submission_template_path = base_data_dir / "39_Test_Dataset" / "sample_submission.csv"

    global_tsfel_segment_feature_names: Optional[List[str]] = None
    logger.info("步驟 1: 生成或加載 TSFEL 片段特徵...")
    train_features_exist = tabular_train_segments_dir.exists() and any(
        f.is_file() for f in tabular_train_segments_dir.iterdir() if f.suffix == ".csv"
    )
    if not train_features_exist:
        global_tsfel_segment_feature_names = data_generate_tsfel_segment_features_from_raw(
            str(train_data_raw_path), str(tabular_train_segments_dir)
        )
    else:
        logger.info(f"從現有目錄 {tabular_train_segments_dir} 加載訓練數據 TSFEL 特徵...")
        try:
            first_csv = next(tabular_train_segments_dir.glob("*.csv"))
            global_tsfel_segment_feature_names = list(pd.read_csv(first_csv, nrows=0).columns)
        except StopIteration:
            logger.warning(f"目錄 {tabular_train_segments_dir} 中沒有 CSV。重新生成。")
            global_tsfel_segment_feature_names = data_generate_tsfel_segment_features_from_raw(
                str(train_data_raw_path), str(tabular_train_segments_dir)
            )
        except Exception as e:
            logger.error(f"讀取特徵名失敗: {e}。重新生成。")
            global_tsfel_segment_feature_names = data_generate_tsfel_segment_features_from_raw(
                str(train_data_raw_path), str(tabular_train_segments_dir)
            )
    test_features_exist = tabular_test_segments_dir.exists() and any(
        f.is_file() for f in tabular_test_segments_dir.iterdir() if f.suffix == ".csv"
    )
    if not test_features_exist:
        data_generate_tsfel_segment_features_from_raw(
            str(test_data_raw_path),
            str(tabular_test_segments_dir),
            global_tsfel_segment_feature_names,
        )
    else:
        logger.info(f"測試數據 TSFEL 特徵已存在於 {tabular_test_segments_dir}。")
    if global_tsfel_segment_feature_names is None or not global_tsfel_segment_feature_names:
        logger.error("未能確定 TSFEL 特徵名。")
        return
    logger.info(f"TSFEL 片段特徵名數量: {len(global_tsfel_segment_feature_names)}")

    logger.info("步驟 2: 讀取訓練資訊...")
    try:
        info_df = pd.read_csv(train_info_path)
    except FileNotFoundError:
        logger.error(f"{train_info_path} 找不到。")
        return
    info_df = info_df.dropna(subset=["player_id"])
    info_df["player_id"] = info_df["player_id"].astype(int)

    logger.info("步驟 3: 載入並聚合 TSFEL 片段特徵...")
    all_aggregated_features_list, all_targets_list, all_player_ids_for_split = [], [], []
    feature_files_train = sorted(list(tabular_train_segments_dir.glob("*.csv")))
    if not feature_files_train:
        logger.error(f"{tabular_train_segments_dir} 中無 TSFEL 特徵檔案。")
        return
    final_aggregated_feature_names: Optional[List[str]] = None
    for file_path in feature_files_train:
        try:
            uid = int(file_path.stem)
            row_info = info_df[info_df["unique_id"] == uid]
        except ValueError:
            continue
        if row_info.empty:
            continue
        try:
            segment_features_df = pd.read_csv(file_path)
        except Exception:
            continue
        if segment_features_df.empty:
            continue
        segment_features_df = segment_features_df.reindex(
            columns=global_tsfel_segment_feature_names, fill_value=np.nan
        )
        aggregated_series = aggregate_segment_features(segment_features_df, AGGREGATION_METHODS_MAP)
        if final_aggregated_feature_names is None and not aggregated_series.empty:
            final_aggregated_feature_names = list(aggregated_series.index)
        if not aggregated_series.empty:
            if final_aggregated_feature_names:
                aggregated_series = aggregated_series.reindex(
                    index=final_aggregated_feature_names, fill_value=np.nan
                )
            all_aggregated_features_list.append(aggregated_series)
            all_targets_list.append(row_info.iloc[0])
            all_player_ids_for_split.append(row_info["player_id"].iloc[0])
    if not all_aggregated_features_list or final_aggregated_feature_names is None:
        logger.error("未能聚合 TSFEL 特徵。")
        return
    X_aggregated_all_df_raw = pd.DataFrame(all_aggregated_features_list).reset_index(drop=True)
    X_aggregated_all_df_raw = X_aggregated_all_df_raw.reindex(
        columns=final_aggregated_feature_names, fill_value=np.nan
    ).astype(np.float32)
    Y_df_all = pd.DataFrame(all_targets_list)[TARGET_COLS].reset_index(drop=True)
    player_ids_all_series = pd.Series(
        all_player_ids_for_split, name="player_id", index=X_aggregated_all_df_raw.index
    )
    logger.info(f"聚合訓練數據形狀: X={X_aggregated_all_df_raw.shape}, Y={Y_df_all.shape}")

    logger.info("\n步驟 3.5: 全局基礎特徵選擇...")
    _, global_selected_feature_names = select_features_basic(
        X_aggregated_all_df_raw.copy(), variance_threshold_val=0.01
    )
    if not global_selected_feature_names:
        logger.error("全局特徵選擇後無特徵。")
        return
    X_data_for_models = (
        X_aggregated_all_df_raw[global_selected_feature_names].copy().astype(np.float32)
    )
    logger.info(
        f"全局選擇後特徵數量: {len(global_selected_feature_names)}. 數據形狀: {X_data_for_models.shape}"
    )

    label_encoders: Dict[str, LabelEncoder] = {col: LabelEncoder() for col in TARGET_COLS}
    for col in TARGET_COLS:
        unique_labels = Y_df_all[col].astype(str).fillna("NaN_placeholder").unique()
        label_encoders[col].fit(unique_labels)
    logger.info(f"'gender' 分佈:\n{Y_df_all['gender'].value_counts(normalize=True)}")
    logger.info(f"'play years' 分佈:\n{Y_df_all['play years'].value_counts(normalize=True)}")
    logger.info(f"'level' 分佈:\n{Y_df_all['level'].value_counts(normalize=True)}")

    logger.info(f"\n步驟 4 & 6: CatBoost 模型訓練與超參數優化...")
    best_models: Dict[str, CatBoostClassifier] = {}
    best_model_params_and_meta: Dict[str, Dict[str, Any]] = {}
    overall_cv_metrics_summary: Dict[str, Dict[str, float]] = {target: {} for target in TARGET_COLS}
    X_data_for_models_reset = X_data_for_models.reset_index(drop=True)
    Y_df_all_reset = Y_df_all.reset_index(drop=True)
    player_ids_all_series_reset = player_ids_all_series.reset_index(drop=True)

    for target_idx, target in enumerate(TARGET_COLS):
        logger.info(
            f"\n--- ({target_idx+1}/{len(TARGET_COLS)}) 開始為目標 '{target}' (CatBoost with TSFEL - GPU v7) ---"
        )
        is_binary_target = target in BINARY_TARGETS
        y_target_str_filled = Y_df_all_reset[target].astype(str).fillna("NaN_placeholder")
        y_target_encoded = label_encoders[target].transform(y_target_str_filled)

        model_path = MODELS_DIR / f"catboost_model_{target}.cbm"
        imputer_path = MODELS_DIR / f"imputer_{target}.joblib"
        scaler_path = MODELS_DIR / f"scaler_{target}.joblib"
        params_and_meta_path = MODELS_DIR / f"best_params_meta_{target}.json"

        if (
            USE_SAVED_MODELS
            and model_path.exists()
            and imputer_path.exists()
            and scaler_path.exists()
            and params_and_meta_path.exists()
        ):
            logger.info(f"為目標 '{target}' 載入已儲存的成果...")
            try:
                final_model = CatBoostClassifier().load_model(str(model_path))
                final_model_imputer = joblib.load(imputer_path)
                final_model_scaler = joblib.load(scaler_path)
                with open(params_and_meta_path, "r") as f:
                    loaded_params_meta = json.load(f)
                best_model_params_and_meta[target] = loaded_params_meta
                overall_cv_metrics_summary[target]["best_cv_score"] = loaded_params_meta.get(
                    "optuna_best_value", np.nan
                )
                best_models[target] = final_model
                best_models[target].final_imputer_ = final_model_imputer
                best_models[target].final_scaler_ = final_model_scaler
                best_models[target].selected_feature_names_ = loaded_params_meta.get(
                    "selected_feature_names", global_selected_feature_names
                )
                logger.info(f"目標 '{target}' 模型已載入。跳過訓練。")
                continue
            except Exception as e:
                logger.error(f"載入 '{target}' 模型失敗: {e}。重新訓練。")

        logger.info(f"為 '{target}' 執行 Optuna...")

        def objective(
            trial: optuna.Trial,
            current_target_obj=target,
            current_X_base_obj=X_data_for_models_reset,
            current_y_enc_obj=y_target_encoded,
            current_pids_obj=player_ids_all_series_reset,
            current_is_binary_obj=is_binary_target,
            current_le_obj=label_encoders[target],
        ) -> float:
            X_for_trial = current_X_base_obj.copy()
            valid_y_indices = ~np.isnan(current_y_enc_obj)
            k_for_folds: Optional[int] = None
            if not np.all(valid_y_indices):
                logger.warning(f"{current_target_obj} y_encoded 含 NaN")
                return 0.0

            catboost_trial_params = {
                "iterations": trial.suggest_int("iterations", 500, 2500, step=200),
                "learning_rate": trial.suggest_float("learning_rate", 5e-4, 0.05, log=True),
                "depth": trial.suggest_int("depth", 3, 6),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-1, 10.0, log=True),
                "border_count": trial.suggest_int("border_count", 32, 64),
                "random_strength": trial.suggest_float("random_strength", 1e-7, 1.0, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
                "loss_function": "Logloss" if current_is_binary_obj else "MultiClass",
                "eval_metric": "Logloss" if current_is_binary_obj else "MultiClass",
                "random_seed": 42,
                "task_type": "GPU",
                "logging_level": "Silent",
                "early_stopping_rounds": trial.suggest_int(
                    "early_stopping_rounds_cb", 40, 100, step=10
                ),
                "add_input_noise": trial.suggest_categorical("add_input_noise", [False, True]),
            }
            if catboost_trial_params["add_input_noise"]:
                catboost_trial_params["noise_level"] = trial.suggest_float(
                    "noise_level", 1e-4, 0.02, log=True
                )
            else:
                catboost_trial_params["noise_level"] = 0.0

            if current_target_obj in ["gender", "play years", "level"]:
                max_k = X_for_trial.shape[1]
                min_k = min(20, max_k) if max_k > 20 else (1 if max_k > 0 else 0)
                if min_k > 0:
                    k_val = trial.suggest_int(f"k_selectkbest_{current_target_obj}", min_k, max_k)
                    catboost_trial_params[f"k_selectkbest_{current_target_obj}"] = k_val
                    # SelectKBest 在下方的 fold 迴圈內只用訓練資料擬合，避免看到驗證 fold 的標籤
                    if 0 < k_val < X_for_trial.shape[1]:
                        k_for_folds = k_val
                else:
                    catboost_trial_params[f"k_selectkbest_{current_target_obj}"] = (
                        X_for_trial.shape[1]
                    )

            if current_target_obj == "gender" and current_is_binary_obj:
                counts = np.bincount(current_y_enc_obj[valid_y_indices])
                if len(counts) == 2 and counts[1] > 0:
                    calculated_spw = counts[0] / counts[1]
                    catboost_trial_params["scale_pos_weight"] = trial.suggest_float(
                        "scale_pos_weight",
                        max(0.1, calculated_spw * 0.5),
                        min(20.0, calculated_spw * 2.0),
                        log=True,
                    )
                else:
                    catboost_trial_params["scale_pos_weight"] = trial.suggest_float(
                        "scale_pos_weight", 0.5, 10.0, log=True
                    )

            if current_target_obj in ["play years", "level"] and not current_is_binary_obj:
                use_cw = trial.suggest_categorical(
                    f"use_class_weights_{current_target_obj}", [False, True]
                )
                catboost_trial_params[f"use_class_weights_{current_target_obj}"] = use_cw

            fold_scores_list = []
            gkf = GroupKFold(n_splits=N_SPLITS_GROUPKFOLD)
            for fold, (tr_idx, va_idx) in enumerate(
                gkf.split(X_for_trial, current_y_enc_obj, groups=current_pids_obj)
            ):
                X_tr, X_va = X_for_trial.iloc[tr_idx].copy(), X_for_trial.iloc[va_idx].copy()
                y_tr, y_va = current_y_enc_obj[tr_idx], current_y_enc_obj[va_idx]
                if k_for_folds is not None:
                    try:
                        X_tr_for_kbest = X_tr.fillna(X_tr.median()).fillna(0)
                        selector_kbest = SelectKBest(f_classif, k=k_for_folds).fit(
                            X_tr_for_kbest, y_tr
                        )
                        kbest_cols = X_tr.columns[selector_kbest.get_support(indices=True)]
                        X_tr, X_va = X_tr[kbest_cols], X_va[kbest_cols]
                    except Exception as e_kbest:
                        logger.warning(
                            f"Fold {fold+1} SelectKBest for {current_target_obj} failed: {e_kbest}. Using all base features."
                        )
                if X_va.empty or y_va.size == 0:
                    return 0.0

                fold_catboost_params = catboost_trial_params.copy()
                if (
                    current_target_obj in ["play years", "level"]
                    and not current_is_binary_obj
                    and fold_catboost_params.get(f"use_class_weights_{current_target_obj}", False)
                ):
                    unique_classes_y_tr_fold = np.unique(y_tr)
                    if len(unique_classes_y_tr_fold) > 1:
                        weights_tr_fold = compute_class_weight(
                            class_weight="balanced", classes=unique_classes_y_tr_fold, y=y_tr
                        )
                        fold_catboost_params["class_weights"] = {
                            cls_idx: w
                            for cls_idx, w in zip(unique_classes_y_tr_fold, weights_tr_fold)
                        }
                    elif "class_weights" in fold_catboost_params:
                        del fold_catboost_params["class_weights"]

                scores = train_evaluate_model_cv_fold(
                    trial,
                    X_tr,
                    y_tr,
                    X_va,
                    y_va,
                    current_target_obj,
                    current_is_binary_obj,
                    current_le_obj,
                    fold,
                    fold_catboost_params,
                )
                fold_scores_list.append(scores)
                running_vals = np.array(
                    [
                        s.get("roc_auc" if current_is_binary_obj else "roc_auc_micro_ovr", np.nan)
                        for s in fold_scores_list
                    ],
                    dtype=float,
                )
                trial.report(
                    float(np.nanmean(running_vals)) if np.any(~np.isnan(running_vals)) else 0.0,
                    step=fold,
                )
                if trial.should_prune():
                    raise optuna.TrialPruned()
            trial.set_user_attr(
                "best_iterations",
                [
                    int(s["best_iteration"])
                    for s in fold_scores_list
                    if s.get("best_iteration") is not None
                ],
            )
            metric_key_primary = "roc_auc" if current_is_binary_obj else "roc_auc_micro_ovr"
            metric_key_secondary = "roc_auc_macro_ovr"
            avg_primary_score = np.nanmean(
                [s.get(metric_key_primary, np.nan) for s in fold_scores_list if s]
            )
            if np.isnan(avg_primary_score) and not current_is_binary_obj:
                avg_secondary_score = np.nanmean(
                    [s.get(metric_key_secondary, np.nan) for s in fold_scores_list if s]
                )
                return avg_secondary_score if not np.isnan(avg_secondary_score) else 0.0
            return avg_primary_score if not np.isnan(avg_primary_score) else 0.0

        study = optuna.create_study(
            direction="maximize",
            pruner=optuna.pruners.MedianPruner(n_startup_trials=7, n_warmup_steps=1),
        )
        study.optimize(objective, n_trials=OPTUNA_N_TRIALS, timeout=3600 * 2.5)
        best_trial = study.best_trial
        logger.info(
            f"'{target}' Optuna Best: Val={best_trial.value:.6f}, Params={best_trial.params}"
        )
        current_params_meta_to_save = best_trial.params.copy()
        current_params_meta_to_save["optuna_best_value"] = best_trial.value

        final_features_for_this_target = global_selected_feature_names
        X_train_final_for_this_target = X_data_for_models_reset.copy()
        if target in ["gender", "play years", "level"]:
            best_k_value = best_trial.params.get(f"k_selectkbest_{target}")
            if (
                best_k_value is not None
                and best_k_value > 0
                and best_k_value <= X_data_for_models_reset.shape[1]
            ):
                logger.info(f"'{target}' Final SelectKBest (k={best_k_value})...")
                X_temp_imputed_final_kbest = X_data_for_models_reset.fillna(
                    X_data_for_models_reset.median()
                ).fillna(0)
                final_selector_kbest = SelectKBest(f_classif, k=best_k_value)
                final_selector_kbest.fit(X_temp_imputed_final_kbest, y_target_encoded)
                final_selected_indices = final_selector_kbest.get_support(indices=True)
                final_features_for_this_target = X_data_for_models_reset.columns[
                    final_selected_indices
                ].tolist()
                X_train_final_for_this_target = X_data_for_models_reset[
                    final_features_for_this_target
                ]
                logger.info(f"'{target}' 最終使用 {len(final_features_for_this_target)} 特徵。")
            else:
                logger.info(f"'{target}' k值無效，用全局特徵。")
        current_params_meta_to_save["selected_feature_names"] = final_features_for_this_target
        best_model_params_and_meta[target] = current_params_meta_to_save

        cb_param_keys_to_remove = [
            "add_input_noise",
            "noise_level",
            f"k_selectkbest_{target}",
            f"use_class_weights_{target}",
            "early_stopping_rounds_cb",
        ]
        final_catboost_params_from_optuna = {
            k: v for k, v in best_trial.params.items() if k not in cb_param_keys_to_remove
        }
        # 最終模型用全部資料訓練、沒有驗證集可做 early stopping，因此以 CV 各 fold 最佳迭代數的平均作為樹的數量
        cv_best_iterations = best_trial.user_attrs.get("best_iterations", [])
        if cv_best_iterations:
            final_catboost_params_from_optuna["iterations"] = (
                int(round(float(np.mean(cv_best_iterations)))) + 1
            )
            logger.info(
                f"'{target}' 最終模型 iterations = {final_catboost_params_from_optuna['iterations']} (CV 各 fold 最佳迭代數平均)"
            )
        current_params_meta_to_save["final_iterations"] = final_catboost_params_from_optuna.get(
            "iterations"
        )
        final_catboost_model_params = {
            "loss_function": "Logloss" if is_binary_target else "MultiClass",
            "eval_metric": "Logloss" if is_binary_target else "MultiClass",
            "task_type": "GPU",
            "random_seed": 42,
            "logging_level": "Silent",
            **final_catboost_params_from_optuna,
        }
        if target == "gender" and is_binary_target and "scale_pos_weight" in best_trial.params:
            final_catboost_model_params["scale_pos_weight"] = best_trial.params["scale_pos_weight"]

        if (
            target in ["play years", "level"]
            and not is_binary_target
            and best_trial.params.get(f"use_class_weights_{target}", False)
        ):
            logger.info(f"為 '{target}' 的最終模型計算並應用 class_weights...")
            unique_classes_final = np.unique(y_target_encoded)
            if len(unique_classes_final) > 1:
                final_weights = compute_class_weight(
                    class_weight="balanced", classes=unique_classes_final, y=y_target_encoded
                )
                final_catboost_model_params["class_weights"] = {
                    cls_idx: w for cls_idx, w in zip(unique_classes_final, final_weights)
                }
                logger.info(
                    f"'{target}' 最終模型 Class weights: {final_catboost_model_params['class_weights']}"
                )

        if "iterations" not in final_catboost_model_params:
            final_catboost_model_params["iterations"] = 800

        logger.info(f"'{target}' 最終模型數據準備 (插補、縮放)...")
        final_model_imputer = KNNImputer(
            n_neighbors=min(
                5,
                (
                    X_train_final_for_this_target.shape[0] - 1
                    if X_train_final_for_this_target.shape[0] > 1
                    else 1
                ),
            )
        )
        X_imputed_arr_final = final_model_imputer.fit_transform(X_train_final_for_this_target)
        X_imputed_final_df = pd.DataFrame(
            X_imputed_arr_final, columns=final_features_for_this_target
        ).astype(np.float32)
        final_model_scaler = MinMaxScaler()
        X_scaled_arr_final = final_model_scaler.fit_transform(X_imputed_final_df)
        X_scaled_final_df = pd.DataFrame(
            X_scaled_arr_final, columns=final_features_for_this_target
        ).astype(np.float32)
        final_X_fit_data_for_model = X_scaled_final_df.copy()
        if (
            best_trial.params.get("add_input_noise", False)
            and best_trial.params.get("noise_level", 0.0) > 0
        ):
            noise_val = best_trial.params["noise_level"]
            noise_array = RNG.normal(0, noise_val, final_X_fit_data_for_model.shape)
            final_X_fit_data_for_model = (final_X_fit_data_for_model + noise_array).astype(
                np.float32
            )

        logger.info(f"'{target}' 訓練最終 CatBoost 模型...")
        final_model = CatBoostClassifier(**final_catboost_model_params)
        if "early_stopping_rounds" in final_catboost_model_params:
            logger.info(
                f"最終模型訓練使用迭代 {final_catboost_model_params.get('iterations')} 並啟用早停。"
            )
        final_model.fit(final_X_fit_data_for_model.values, y_target_encoded, verbose=0)
        best_models[target] = final_model
        best_models[target].final_imputer_ = final_model_imputer
        best_models[target].final_scaler_ = final_model_scaler
        best_models[target].selected_feature_names_ = final_features_for_this_target
        try:
            logger.info(f"為 '{target}' 儲存成果...")
            final_model.save_model(str(model_path))
            joblib.dump(final_model_imputer, imputer_path)
            joblib.dump(final_model_scaler, scaler_path)
            with open(params_and_meta_path, "w") as f:
                json.dump(best_model_params_and_meta[target], f, indent=4)
            logger.info(f"'{target}' 成果已儲存。")
        except Exception as e:
            logger.error(f"儲存 '{target}' 成果失敗: {e}")
        logger.info(f"'{target}' 最終模型訓練完成。")
        overall_cv_metrics_summary[target]["best_cv_score"] = best_trial.value

    logger.info("\n--- Optuna 最佳 CV 分數摘要 (v7) ---")
    for target_name_summary, metrics_summary in overall_cv_metrics_summary.items():
        logger.info(
            f"目標 '{target_name_summary}': Best CV Score = {metrics_summary.get('best_cv_score', np.nan):.6f}"
        )

    # --- 步驟 7: 測試集預測 (概率格式) ---
    logger.info("\n步驟 7: 在實際測試集上預測並生成概率提交檔案 (v7)...")
    try:
        submission_df_template = pd.read_csv(submission_template_path)
        test_unique_ids_ordered = submission_df_template["unique_id"].tolist()
    except FileNotFoundError:
        logger.error(f"提交模板 {submission_template_path} 找不到。")
        return

    all_predictions_list = []

    test_feature_files = sorted(list(tabular_test_segments_dir.glob("*.csv")))

    # 根據 sample_submission.csv 的欄位順序確定期望的輸出欄位
    # 這假設 sample_submission.csv 已經包含了所有需要的概率欄位名
    expected_submission_cols_final = list(submission_df_template.columns)
    if "unique_id" not in expected_submission_cols_final:
        logger.error("提交模板 sample_submission.csv 中缺少 'unique_id' 欄位。")
        return

    for uid_test in test_unique_ids_ordered:
        uid_predictions = {"unique_id": uid_test}
        file_path_test = tabular_test_segments_dir / f"{uid_test}.csv"

        # 為此 UID 初始化所有期望的概率欄位為預設值
        for col_name in expected_submission_cols_final:
            if col_name != "unique_id":
                # 根據欄位名判斷是二元還是多分類，並設定合理的預設概率
                is_binary_col = False
                target_for_col = ""
                for bt in BINARY_TARGETS:
                    if col_name == bt:  # 精確匹配二元目標名
                        is_binary_col = True
                        target_for_col = bt
                        break
                if not is_binary_col:
                    for mct in TARGET_COLS:
                        if not (mct in BINARY_TARGETS) and col_name.startswith(f"{mct}_"):
                            target_for_col = mct
                            break

                if is_binary_col:
                    uid_predictions[col_name] = 0.5
                elif target_for_col:  # 多分類
                    num_classes_default = len(label_encoders[target_for_col].classes_)
                    uid_predictions[col_name] = (
                        1.0 / num_classes_default if num_classes_default > 0 else 0.0
                    )
                else:  # 未知欄位 (理論上不應發生，如果 expected_submission_cols_final 來自模板)
                    uid_predictions[col_name] = 0.0

        if not file_path_test.exists():
            logger.warning(f"測試特徵檔案 {file_path_test.name} 找不到。將使用預設概率。")
            all_predictions_list.append(uid_predictions)
            continue
        try:
            test_segment_features_df = pd.read_csv(file_path_test)
        except Exception as e_read_test_seg:
            logger.warning(
                f"讀取測試片段 {file_path_test.name} 失敗: {e_read_test_seg}。使用預設概率。"
            )
            all_predictions_list.append(uid_predictions)
            continue
        if test_segment_features_df.empty:
            logger.warning(f"{file_path_test.name} 為空。使用預設概率。")
            all_predictions_list.append(uid_predictions)
            continue

        test_segment_features_df = test_segment_features_df.reindex(
            columns=global_tsfel_segment_feature_names, fill_value=np.nan
        )
        test_aggregated_series = aggregate_segment_features(
            test_segment_features_df, AGGREGATION_METHODS_MAP
        )
        if test_aggregated_series.empty:
            logger.warning(f"UID {uid_test} 聚合特徵為空。使用預設概率。")
            all_predictions_list.append(uid_predictions)
            continue

        X_test_single_aggregated_df = pd.DataFrame([test_aggregated_series]).reindex(
            columns=final_aggregated_feature_names, fill_value=np.nan
        )

        for target_pred_loop in TARGET_COLS:
            if target_pred_loop not in best_models or not hasattr(
                best_models[target_pred_loop], "selected_feature_names_"
            ):
                logger.error(
                    f"{target_pred_loop} 模型或選定特徵名不存在。跳過此目標預測 (將使用預設概率)。"
                )
                continue

            model_to_use = best_models[target_pred_loop]
            target_specific_features = model_to_use.selected_feature_names_
            try:
                X_test_target_specific_df = X_test_single_aggregated_df[
                    target_specific_features
                ].astype(np.float32)
            except KeyError as e_key:
                logger.error(
                    f"為目標 {target_pred_loop} 選擇特徵時出錯 (KeyError: {e_key})。跳過此目標。"
                )
                continue
            imputer_for_test = model_to_use.final_imputer_
            scaler_for_test = model_to_use.final_scaler_
            X_test_imputed_arr = imputer_for_test.transform(X_test_target_specific_df)
            X_test_imputed_df = pd.DataFrame(
                X_test_imputed_arr, columns=target_specific_features
            ).astype(np.float32)
            X_test_scaled_arr = scaler_for_test.transform(X_test_imputed_df)
            X_test_scaled_df = pd.DataFrame(
                X_test_scaled_arr, columns=target_specific_features
            ).astype(np.float32)
            final_X_test_data = X_test_scaled_df.copy()
            # 推論時不對測試資料加雜訊：input noise 只是訓練時的正則化，加在測試資料上會讓預測帶有隨機性

            pred_probas = model_to_use.predict_proba(final_X_test_data.values)[0]
            le = label_encoders[target_pred_loop]
            if target_pred_loop in BINARY_TARGETS:
                positive_class_original_label = None
                if target_pred_loop == "gender":
                    positive_class_original_label = "1"  # '1' 代表男生
                elif target_pred_loop == "hold racket handed":
                    positive_class_original_label = "1"  # '1' 代表右手

                if positive_class_original_label:
                    try:
                        positive_class_idx = list(le.classes_).index(positive_class_original_label)
                        uid_predictions[target_pred_loop] = pred_probas[positive_class_idx]
                    except ValueError:
                        logger.warning(
                            f"目標 {target_pred_loop} 正類原始標籤 '{positive_class_original_label}' 未在 LE classes ({le.classes_}) 中。將嘗試使用索引1 (如果存在)。"
                        )
                        if len(pred_probas) > 1:
                            uid_predictions[target_pred_loop] = pred_probas[
                                1
                            ]  # 後備：使用索引1的概率
                        elif len(pred_probas) == 1:
                            uid_predictions[target_pred_loop] = pred_probas[0]  # 如果只有一列概率
                        else:
                            uid_predictions[target_pred_loop] = 0.5  # 最終後備
                    except IndexError:
                        logger.error(f"目標 {target_pred_loop} 預測概率長度不足。將使用預設0.5。")
                        uid_predictions[target_pred_loop] = 0.5
                else:
                    logger.warning(f"目標 {target_pred_loop} 未定義正類標籤。使用預設概率 0.5。")
                    uid_predictions[target_pred_loop] = 0.5
            else:
                for class_idx, class_name_original in enumerate(le.classes_):
                    if class_name_original == "NaN_placeholder":
                        continue
                    # 確保 class_name_original 是字串，以匹配提交格式中的 '0', '1', '2' 等
                    submission_col_name = f"{target_pred_loop}_{str(class_name_original)}"
                    if submission_col_name in uid_predictions:  # 確保只填充期望的欄位
                        uid_predictions[submission_col_name] = pred_probas[class_idx]
                    else:
                        logger.warning(
                            f"欄位 {submission_col_name} 不在期望的提交欄位中，跳過填充。"
                        )

        all_predictions_list.append(uid_predictions)

    final_submission_df = pd.DataFrame(all_predictions_list)
    # 使用 submission_df_template 的 unique_id 和欄位順序來確保最終格式
    # 首先，確保 final_submission_df 包含所有 unique_id
    final_submission_df = pd.merge(
        submission_df_template[["unique_id"]], final_submission_df, on="unique_id", how="left"
    )

    # 填充可能因 merge 產生的 NaN，並確保所有期望的欄位都存在
    for col in expected_submission_cols_final:
        if col == "unique_id":
            continue
        if col not in final_submission_df.columns:  # 如果某個概率列完全缺失
            final_submission_df[col] = 0.0  # 或其他合理的預設
            logger.warning(f"提交 DataFrame 中缺失欄位 {col}，已用 0.0 填充。")

        # 填充該列中可能存在的 NaN
        is_binary_col_fill = False
        target_for_col_fill = ""
        for bt_fill in BINARY_TARGETS:
            if col == bt_fill:
                is_binary_col_fill = True
                target_for_col_fill = bt_fill
                break
        if not is_binary_col_fill:
            for mct_fill in TARGET_COLS:
                if not (mct_fill in BINARY_TARGETS) and col.startswith(f"{mct_fill}_"):
                    target_for_col_fill = mct_fill
                    break

        if is_binary_col_fill:
            final_submission_df[col] = final_submission_df[col].fillna(0.5)
        elif target_for_col_fill:
            num_classes_default_fill = len(label_encoders[target_for_col_fill].classes_)
            default_proba_fill_val = (
                1.0 / num_classes_default_fill if num_classes_default_fill > 0 else 0.0
            )
            final_submission_df[col] = final_submission_df[col].fillna(default_proba_fill_val)
        else:  # 未知欄位，不太可能發生，因為 expected_submission_cols_final 來自模板
            final_submission_df[col] = final_submission_df[col].fillna(0.0)

    final_submission_df = final_submission_df[expected_submission_cols_final]  # 確保最終的列順序

    logger.info("清理提交數據中各欄位的內部換行符...")
    for col_to_clean in final_submission_df.columns:
        if final_submission_df[col_to_clean].dtype == "object":
            final_submission_df[col_to_clean] = (
                final_submission_df[col_to_clean]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.replace("\r", " ", regex=False)
            )

    submission_path = base_data_dir / "submission_catboost_tsfel_gpu_v7.csv"
    try:
        final_submission_df.to_csv(
            submission_path, index=False, float_format="%.4f", lineterminator="\n", encoding="utf-8"
        )
        logger.info(f"預測完成，提交檔案已保存至: {submission_path}")
    except TypeError:
        logger.warning(
            "當前 Pandas 版本不支持 'line_terminator' 參數。將不使用該參數儲存。請考慮升級 Pandas。"
        )
        final_submission_df.to_csv(
            submission_path, index=False, float_format="%.4f", encoding="utf-8"
        )
        logger.info(f"預測完成 (無 line_terminator)，提交檔案已保存至: {submission_path}")
    except Exception as e_final_save:
        logger.error(f"保存最終提交檔案失敗: {e_final_save}")

    logger.info("腳本執行完畢 (v7 with Advanced Optimizations & Probability Submission)。")

## 6. Run

對應 README 的 **Setup / Running**。執行下方 cell 即依序跑完整條 pipeline。

- 先將 `39_Training_Dataset/` 與 `39_Test_Dataset/` 放在 repo 根目錄。
- 中間 TSFEL 特徵與模型 artifact 會自動 cache，重跑時略過耗時步驟。
- 若要 **重新訓練**：將第 1 區塊的 `USE_SAVED_MODELS` 設為 `False`，或刪除 `trained_models_catboost_v7/` 與兩個 `tabular_data_*_v7/` 資料夾。


In [ ]:
if __name__ == "__main__":
    main()